# Action Recognition Thesis — Kaggle Training Notebook

**Before running:** In the right-hand panel:
1. **Settings → Accelerator → GPU T4 x2** (free, ~30 GPU-hrs/week)
2. **Add Data** → search `ucf101-action-recognition` → add the `matthewjansen/ucf101-action-recognition` dataset
   - If that exact slug isn't found, search 'UCF101' and pick one that has both a `UCF-101/` video folder and split files (`classInd.txt`, `trainlist01.txt`)

Then **Run All** (Run → Run All Cells).

---
This notebook fine-tunes all three architectures on UCF101 split 1 and benchmarks them.
Results are saved to `results/` and to `/kaggle/output/` for download.

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
# torch/torchvision are pre-installed on Kaggle; we only need these extras.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'decord', 'fvcore', 'scikit-learn', 'huggingface_hub'
], check=True)
print('deps installed')

In [ ]:
# ── Cell 2: Clone the thesis repo ────────────────────────────────────────────
import subprocess, sys, os

REPO = 'https://github.com/AndreiBrehuescu/action-recognition-thesis.git'
WORK = '/kaggle/working/thesis'

if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, WORK], check=True)
else:
    subprocess.run(['git', '-C', WORK, 'pull', '--ff-only'], check=True)

if WORK not in sys.path:
    sys.path.insert(0, WORK)
os.chdir(WORK)
print(f'repo at {WORK}')

In [ ]:
# ── Cell 3: Verify GPU and locate the UCF101 data ───────────────────────────
import torch
from pathlib import Path

print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        gb = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({gb:.1f} GB)')

print()
input_dir = Path('/kaggle/input')
print('Kaggle input datasets:')
for p in sorted(input_dir.iterdir()):
    print(f'  {p}')

# Auto-detect the UCF101 dataset path
DATA_ROOT = None
for p in input_dir.iterdir():
    if list(p.rglob('classInd.txt')):
        DATA_ROOT = str(p)
        print(f'\nDetected UCF101 at: {DATA_ROOT}')
        break

if DATA_ROOT is None:
    raise RuntimeError(
        'UCF101 dataset not found in /kaggle/input. '
        'Add the dataset in the right-hand panel (see notebook header).'
    )

In [ ]:
# ── Cell 4: Quick sanity-check — one mini-batch through each model ───────────
import torch
from src.datasets import build_dataset
from src.models import build_model
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load a tiny slice of the training set
ds, num_classes = build_dataset('ucf101', DATA_ROOT, split=1, train=True)
loader = DataLoader(ds, batch_size=4, shuffle=True, num_workers=2)
clips, labels = next(iter(loader))
print(f'Clip tensor: {clips.shape}  labels: {labels}')
print(f'num_classes: {num_classes}')

for name in ['resnet50_tsn', 'r2plus1d_18', 'videomae']:
    m = build_model(name, num_classes).to(device)
    with torch.no_grad():
        out = m(clips.to(device))
    print(f'{name}: output {out.shape}  OK')
    del m
    torch.cuda.empty_cache()

In [ ]:
# ── Cell 5: Train VideoMAE (Transformer) — 10 epochs ────────────────────────
# Slowest per-epoch (~12-18 min on T4) but highest accuracy ceiling.
# Kinetics-pretrained, fine-tuned on UCF101.
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'src.train',
    '--model', 'videomae',
    '--dataset', 'ucf101',
    '--data-root', DATA_ROOT,
    '--epochs', '10',
    '--batch-size', '8',
    '--num-workers', '2',
], check=True)

In [ ]:
# ── Cell 6: Train R(2+1)D-18 (3D-CNN) — 10 epochs ───────────────────────────
# Kinetics400 weights. ~3-5 min/epoch on T4.
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'src.train',
    '--model', 'r2plus1d_18',
    '--dataset', 'ucf101',
    '--data-root', DATA_ROOT,
    '--epochs', '10',
    '--batch-size', '16',
    '--num-workers', '2',
], check=True)

In [ ]:
# ── Cell 7: Train ResNet-50 TSN (2D-CNN) — 10 epochs ────────────────────────
# ImageNet pretrained, per-frame + temporal averaging. Fastest; ~1-2 min/epoch.
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'src.train',
    '--model', 'resnet50_tsn',
    '--dataset', 'ucf101',
    '--data-root', DATA_ROOT,
    '--epochs', '10',
    '--batch-size', '32',
    '--num-workers', '2',
], check=True)

In [ ]:
# ── Cell 8: Efficiency benchmark (params / GFLOPs / latency / memory) ────────
# Runs on the GPU with a synthetic batch — no data loading needed.
import subprocess, sys

for model_name in ['resnet50_tsn', 'r2plus1d_18', 'videomae']:
    print(f'\n=== Benchmarking {model_name} ===')
    subprocess.run([
        sys.executable, '-m', 'src.benchmark',
        '--model', model_name,
        '--dataset', 'ucf101',
        '--data-root', DATA_ROOT,
    ], check=True)

In [ ]:
# ── Cell 9: Summarise results ────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

results_dir = Path('/kaggle/working/thesis/results')

# Per-model best accuracy
print('=== Training Results (best epoch highlighted) ===')
for f in sorted(results_dir.glob('*_ucf101_train.csv')):
    df = pd.read_csv(f)
    best = df.loc[df['top1'].astype(float).idxmax()]
    print(f"\n{f.stem}")
    print(df.to_string(index=False))
    print(f"  ↑ best top-1: {best['top1']}% @ epoch {int(best['epoch'])}")

# Efficiency summary
eff = results_dir / 'efficiency.csv'
if eff.exists():
    print('\n=== Efficiency Comparison ===')
    print(pd.read_csv(eff).to_string(index=False))

In [ ]:
# ── Cell 10: Pareto scatter — accuracy vs GFLOPs ────────────────────────────
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

results_dir = Path('/kaggle/working/thesis/results')
eff = pd.read_csv(results_dir / 'efficiency.csv')

# Grab best top-1 per model from train logs
acc = {}
for f in results_dir.glob('*_ucf101_train.csv'):
    model = f.stem.replace('_ucf101_train', '')
    df = pd.read_csv(f)
    acc[model] = df['top1'].astype(float).max()

eff['top1'] = eff['model'].map(acc)

fig, ax = plt.subplots(figsize=(8, 5))
colors = {'resnet50_tsn': '#4C72B0', 'r2plus1d_18': '#DD8452', 'videomae': '#55A868'}
for _, row in eff.iterrows():
    c = colors.get(row['model'], 'gray')
    ax.scatter(row['gflops'], row['top1'], s=row['params_M'] * 2, color=c,
               alpha=0.85, edgecolors='k', linewidths=0.5, zorder=3)
    ax.annotate(row['model'], (row['gflops'], row['top1']),
                textcoords='offset points', xytext=(6, 4), fontsize=9)

ax.set_xlabel('GFLOPs / clip', fontsize=11)
ax.set_ylabel('UCF101 Top-1 Accuracy (%)', fontsize=11)
ax.set_title('Accuracy vs Computational Cost\n(bubble size ∝ parameter count)', fontsize=12)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(str(results_dir / 'pareto_scatter.pdf'), bbox_inches='tight')
fig.savefig(str(results_dir / 'pareto_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved pareto_scatter.pdf + .png')

In [ ]:
# ── Cell 11: Copy results to Kaggle output (persisted after session ends) ────
import shutil, os
from pathlib import Path

src = Path('/kaggle/working/thesis/results')
dst = Path('/kaggle/working/output')
dst.mkdir(exist_ok=True)

for f in src.rglob('*'):
    if f.is_file():
        rel = f.relative_to(src)
        target = dst / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, target)

print('Output files:')
for f in sorted(dst.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(dst)}  ({f.stat().st_size/1024:.0f} KB)')